# Bagging Regressor and Boosting

**Additional learning example:** This notebook was added to extend the original Random Forest and Bagging work. It is new supporting code, not part of the original uploaded notebook. It compares a Decision Tree regressor, Bagging Regressor, AdaBoost Regressor, and Gradient Boosting Regressor using Scikit-learn's Diabetes dataset.

## 1. Load the dataset and split the data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 2. Train the models

The same training and test split is used for every model to make the comparison more consistent. Tree-based models generally do not require feature standardization.

In [ ]:
models = {
    "Decision Tree": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Bagging Regressor": BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "AdaBoost Regressor": AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3, random_state=42),
        n_estimators=100,
        random_state=42
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    )
}

predictions = {}
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    predictions[name] = estimator.predict(X_test)
    print(f"{name}: trained")

## 3. Compare evaluation metrics

MAE and RMSE are in the original target units; lower values are better. R² describes the fraction of target variance explained relative to a mean-prediction baseline; higher is generally better.

In [ ]:
results = []
for name, y_pred in predictions.items():
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

## 4. Interpret the comparison

- **Decision Tree:** a single tree is easy to inspect but can be sensitive to the training sample.
- **Bagging Regressor:** aggregates bootstrap-trained estimators and can reduce variance.
- **AdaBoost Regressor:** sequentially adjusts emphasis to improve the ensemble according to the boosting algorithm.
- **Gradient Boosting Regressor:** builds an additive model in stages to reduce prediction errors.

The best method depends on the dataset, hyperparameters, and selected metric. These results are a small demonstration, not a universal ranking. Use cross-validation for model selection before drawing conclusions.